## Data Cleaning


In [ ]:
from datasets import load_dataset
import pandas as pd
import re
import os

/home/ashtondy/School/cs/cs429/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


ModuleNotFoundError: No module named 'datasets'

In [ ]:
dataset = load_dataset("Ahmad0067/MedSynth")
data = dataset["train"]

print(len(data))
print(data[0])

In [ ]:
for i in range(3):
    print("---- SAMPLE ----")
    print(data[i]["Dialogue"])
    print("\nSOAP:", data[i][" Note"])
    print("\n")

In [ ]:
import re

def clean_transcript(text):
    if not text:
        return None

    text = text.strip()
    text = re.sub(r'\s+', ' ', text)

    text = text.replace("Doctor:", "[Doctor]:")
    text = text.replace("Patient:", "[Patient]:")

    MAX_CHARS = 4000
    return text[:MAX_CHARS]


def clean_soap_note(text):
    if not text:
        return None

    text = text.replace("**", "")
    text = text.replace("##", "")

    text = text.strip()
    text = re.sub(r'\s+', ' ', text)

    return text

In [ ]:
sample = data[0]

raw_transcript = sample["Dialogue"]
raw_soap = sample[" Note"]

clean_trans = clean_transcript(raw_transcript)
clean_soap = clean_soap_note(raw_soap)

print("===== RAW TRANSCRIPT =====\n")
print(raw_transcript)

print("\n===== CLEAN TRANSCRIPT =====\n")
print(clean_trans)

print("\n============================\n")

print("===== RAW SOAP =====\n")
print(raw_soap)

print("\n===== CLEAN SOAP =====\n")
print(clean_soap)

In [ ]:
def clean_transcript(text):
    if not text:
        return None

    # normalize spaces BUT keep line breaks
    text = text.strip()

    # standardize speakers
    text = text.replace("[doctor]:", "[Doctor]:")
    text = text.replace("[patient]:", "[Patient]:")

    return text


def clean_soap_note(text):
    if not text:
        return None

    # remove markdown symbols
    text = text.replace("**", "")
    text = text.replace("##", "")

    # remove only section numbers like "1. ", "2. "
    text = re.sub(r'^\d+\.\s*', '', text, flags=re.MULTILINE)

    # normalize spacing but KEEP line breaks
    text = text.strip()

    return text

In [ ]:
sample = data[0]

raw_transcript = sample["Dialogue"]
raw_soap = sample[" Note"]

clean_trans = clean_transcript(raw_transcript)
clean_soap = clean_soap_note(raw_soap)

print("===== RAW TRANSCRIPT =====\n")
print(raw_transcript)

print("\n===== CLEAN TRANSCRIPT =====\n")
print(clean_trans)

print("\n============================\n")

print("===== RAW SOAP =====\n")
print(raw_soap)

print("\n===== CLEAN SOAP =====\n")
print(clean_soap)

# Check for NULL / Missing Values

In [ ]:
cleaned_data = []

for sample in data:
    transcript = clean_transcript(sample["Dialogue"])
    soap = clean_soap_note(sample[" Note"])  

    if not transcript or not soap:
        continue

    cleaned_data.append({
        "transcript": transcript,
        "ground_truth": soap
    })

print("Total cleaned samples:", len(cleaned_data))

In [ ]:
import pandas as pd

df = pd.DataFrame(cleaned_data)
print(df.head())

In [ ]:
empty_transcripts = df[df["transcript"].str.strip() == ""]
empty_soap = df[df["ground_truth"].str.strip() == ""]

print("Empty transcripts:", len(empty_transcripts))
print("Empty SOAP:", len(empty_soap))

In [ ]:
print("Total duplicate rows:", df.duplicated().sum())

In [ ]:
dup_transcripts = df.duplicated(subset=["transcript"]).sum()
print("Duplicate transcripts:", dup_transcripts)

In [ ]:
df = df.drop_duplicates(subset=["transcript"])

print("After removing duplicates:", len(df))

In [ ]:
df["length"] = df["transcript"].apply(len)

print(df["length"].describe())

In [ ]:
data = data.rename_column(" Note", "Note")

In [ ]:
for i in range(3):
    print("====== SAMPLE", i, "======\n")
    
    print("TRANSCRIPT:\n")
    print(df.iloc[i]["transcript"])
    
    print("\nSOAP:\n")
    print(df.iloc[i]["ground_truth"])
    
    print("\n" + "="*50 + "\n")

In [ ]:
import random

for _ in range(3):
    idx = random.randint(0, len(df)-1)
    
    print("====== RANDOM SAMPLE ======\n")
    
    print("TRANSCRIPT:\n")
    print(df.iloc[idx]["transcript"][:500])  # preview
    
    print("\nSOAP:\n")
    print(df.iloc[idx]["ground_truth"][:500])
    
    print("\n" + "="*50 + "\n")

In [ ]:
def clean_transcript(text):
    if not text:
        return None

    text = text.strip()

    # normalize all variations
    text = text.replace("[doctor]", "[Doctor]")
    text = text.replace("[patient]", "[Patient]")

    text = text.replace("[doctor]:", "[Doctor]:")
    text = text.replace("[patient]:", "[Patient]:")

    return text

In [ ]:
def normalize_sections(text):
    text = re.sub(r'#\s*SUBJECTIVE', 'Subjective:', text, flags=re.IGNORECASE)
    text = re.sub(r'#\s*OBJECTIVE', 'Objective:', text, flags=re.IGNORECASE)
    text = re.sub(r'#\s*ASSESSMENT', 'Assessment:', text, flags=re.IGNORECASE)
    text = re.sub(r'#\s*PLAN', 'Plan:', text, flags=re.IGNORECASE)

    text = re.sub(r'#\s*\d+\.\s*Subjective:', 'Subjective:', text, flags=re.IGNORECASE)
    text = re.sub(r'#\s*\d+\.\s*Objective:', 'Objective:', text, flags=re.IGNORECASE)

    return text

In [ ]:
def clean_soap_note(text):
    if not text:
        return None

    text = text.replace("**", "")
    text = text.replace("##", "")

    text = text.replace("\\n", "\n")

    text = re.sub(r'^\d+\.\s*', '', text, flags=re.MULTILINE)

    text = normalize_sections(text)

    return text.strip()

In [ ]:
for i in range(2):
    print("="*60)
    print(f"SAMPLE {i}")
    print("="*60)
    
    print("\n--- TRANSCRIPT ---\n")
    print(df.iloc[i]["transcript"])
    
    print("\n--- SOAP NOTE ---\n")
    print(df.iloc[i]["ground_truth"])
    
    print("\n\n")

In [ ]:
import random

for _ in range(2):
    idx = random.randint(0, len(df)-1)
    
    print("="*60)
    print(f"RANDOM SAMPLE {idx}")
    print("="*60)
    
    print("\n--- TRANSCRIPT ---\n")
    print(df.iloc[idx]["transcript"])
    
    print("\n--- SOAP NOTE ---\n")
    print(df.iloc[idx]["ground_truth"])
    
    print("\n\n")

In [ ]:
os.makedirs("data", exist_ok=True)

df.to_json("data/clean_medsynth_final.json", orient="records", indent=2)

In [ ]:
df.to_csv("data/clean_medsynth_final.csv", index=False)